# Colab Rerun 3: Build Corrected Panel and Final Sample Flow

Use this notebook when the outputs from notebook 03 are too large to download locally.

It reads the yearly screened TRC and ERA5-Land exports directly from Google Drive,
rebuilds the corrected merged panel in Colab, applies full-period retention,
and defines the corrected `high_veg` subgroup.


## Expected inputs on Drive

- `/content/drive/MyDrive/GEE_exports/UCDB_TRC_*_screened_fast.csv`
- `/content/drive/MyDrive/GEE_exports/ERA5L_daily_*_screened_core.csv`
- `/content/drive/MyDrive/UCDB_TRC_2025_city_retention_all11422.csv`

## Main outputs on Drive

- `nbs_rerun_repair/outputs/TRC_threshold_analysis_panel_anom_corrected.parquet`
- `nbs_rerun_repair/outputs/city_retention_fullperiod_corrected.csv`
- `nbs_rerun_repair/outputs/sample_flow_corrected.csv`
- `nbs_rerun_repair/outputs/panel_city_base_corrected.csv`

## Runtime note

A high-RAM Colab runtime is recommended. This notebook processes the yearly files one-by-one
and writes intermediate parquet files to Drive so you can restart without losing all progress.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyarrow


In [ ]:
from pathlib import Path
from glob import glob
import gc
import re
import pandas as pd
import numpy as np

DRIVE_ROOT = Path("/content/drive/MyDrive")
EXPORT_FOLDER = DRIVE_ROOT / "GEE_exports"
PROJECT_FOLDER = DRIVE_ROOT / "nbs_rerun_repair"
OUT = PROJECT_FOLDER / "outputs"
INTERMEDIATE = OUT / "intermediate"

for path in [PROJECT_FOLDER, OUT, INTERMEDIATE]:
    path.mkdir(parents=True, exist_ok=True)

TRC_GLOB = str(EXPORT_FOLDER / "UCDB_TRC_*_screened_fast.csv")
ERA_GLOB = str(EXPORT_FOLDER / "ERA5L_daily_*_screened_core.csv")
RETENTION_2025 = DRIVE_ROOT / "UCDB_TRC_2025_city_retention_all11422.csv"

MIN_PIXELS = 5
FULL_PERIOD_FRAC = 0.50
WRITE_PANEL_CSV_GZ = False


In [ ]:
trc_files = sorted(glob(TRC_GLOB))
era_files = sorted(glob(ERA_GLOB))

print("TRC files found:", len(trc_files))
print("ERA files found:", len(era_files))
print("Retention file exists:", RETENTION_2025.exists(), RETENTION_2025)

if len(trc_files) == 0 or len(era_files) == 0:
    raise FileNotFoundError(
        "Missing yearly screened exports on Drive. Confirm notebook 03 finished and the files are in MyDrive/GEE_exports."
    )
if not RETENTION_2025.exists():
    raise FileNotFoundError(
        "Missing UCDB_TRC_2025_city_retention_all11422.csv on Drive. Confirm notebook 02 finished."
    )


In [ ]:
def infer_year(path_str):
    match = re.search(r"(19|20)\d{2}", Path(path_str).name)
    if not match:
        raise ValueError(f"Could not infer year from filename: {path_str}")
    return int(match.group(0))


def process_trc_file(path_str):
    year = infer_year(path_str)
    out_path = INTERMEDIATE / f"trc_pentad_{year}.parquet"
    if out_path.exists():
        print(f"Skipping TRC {year}, already processed.")
        return {"year": year, "source_rows": np.nan, "pentad_rows": np.nan, "status": "cached"}

    df = pd.read_csv(path_str, dtype={"uc_id": str})
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year
    df["step8"] = df.groupby(["uc_id", "unit", "year"]).cumcount() + 1

    df["dT_day_qc"] = np.where(
        (df["n_built_day"] >= MIN_PIXELS) & (df["n_veg_day"] >= MIN_PIXELS),
        df["dT_day"],
        np.nan
    )
    df["dT_night_qc"] = np.where(
        (df["n_built_night"] >= MIN_PIXELS) & (df["n_veg_night"] >= MIN_PIXELS),
        df["dT_night"],
        np.nan
    )

    df["dT_day_qc"] = df["dT_day_qc"].where(df["dT_day_qc"].between(-15, 15))
    df["dT_night_qc"] = df["dT_night_qc"].where(df["dT_night_qc"].between(-10, 10))

    trc_p = (
        df.groupby(["uc_id", "unit", "year", "step8"], as_index=False)
          .agg(
              date_min=("date", "min"),
              date_max=("date", "max"),
              dT_day=("dT_day_qc", "mean"),
              dT_night=("dT_night_qc", "mean"),
              tb_day=("tb_day", "mean"),
              tb_night=("tb_night", "mean"),
              tv_day=("tv_day", "mean"),
              tv_night=("tv_night", "mean"),
              n_obs_day=("dT_day_qc", lambda x: x.notna().sum()),
              n_obs_night=("dT_night_qc", lambda x: x.notna().sum()),
              n_built_day=("n_built_day", "mean"),
              n_built_night=("n_built_night", "mean"),
              n_veg_day=("n_veg_day", "mean"),
              n_veg_night=("n_veg_night", "mean"),
          )
    )

    trc_p.to_parquet(out_path, index=False)
    result = {"year": year, "source_rows": len(df), "pentad_rows": len(trc_p), "status": "written"}
    del df, trc_p
    gc.collect()
    return result


trc_log = pd.DataFrame([process_trc_file(path) for path in trc_files]).sort_values("year")
trc_log.to_csv(OUT / "trc_yearly_processing_log.csv", index=False)
trc_log


In [ ]:
def sat_vp(temp_c):
    return 0.6108 * np.exp((17.27 * temp_c) / (temp_c + 237.3))


def process_era_file(path_str):
    year = infer_year(path_str)
    out_path = INTERMEDIATE / f"era_pentad_{year}.parquet"
    if out_path.exists():
        print(f"Skipping ERA {year}, already processed.")
        return {"year": year, "source_rows": np.nan, "pentad_rows": np.nan, "status": "cached"}

    df = pd.read_csv(path_str, dtype={"uc_id": str})
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year
    df["doy"] = df["date"].dt.dayofyear
    df["pentad"] = ((df["doy"] - 1) // 5) + 1

    df["t2m_c"] = df["temperature_2m"] - 273.15
    df["td2m_c"] = df["dewpoint_temperature_2m"] - 273.15
    df["skin_c"] = df["skin_temperature"] - 273.15

    df["es"] = sat_vp(df["t2m_c"])
    df["ea"] = sat_vp(df["td2m_c"])
    df["vpd_kpa"] = df["es"] - df["ea"]

    era_p = (
        df.groupby(["uc_id", "year", "pentad"], as_index=False)
          .agg(
              t2m_c=("t2m_c", "mean"),
              td2m_c=("td2m_c", "mean"),
              skin_c=("skin_c", "mean"),
              rzsm=("rzsm", "mean"),
              vpd_kpa=("vpd_kpa", "mean"),
              pr_sum=("total_precipitation_sum", "sum"),
          )
    )

    era_p.to_parquet(out_path, index=False)
    result = {"year": year, "source_rows": len(df), "pentad_rows": len(era_p), "status": "written"}
    del df, era_p
    gc.collect()
    return result


era_log = pd.DataFrame([process_era_file(path) for path in era_files]).sort_values("year")
era_log.to_csv(OUT / "era_yearly_processing_log.csv", index=False)
era_log


In [ ]:
trc_p_files = sorted((INTERMEDIATE).glob("trc_pentad_*.parquet"))
era_p_files = sorted((INTERMEDIATE).glob("era_pentad_*.parquet"))

trc_p = pd.concat([pd.read_parquet(path) for path in trc_p_files], ignore_index=True)
era_p = pd.concat([pd.read_parquet(path) for path in era_p_files], ignore_index=True)

print("Aggregated TRC shape:", trc_p.shape)
print("Aggregated ERA shape:", era_p.shape)


In [ ]:
step_lookup = (
    trc_p[["year", "step8", "date_min"]]
    .drop_duplicates()
    .sort_values(["year", "step8"])
)
step_lookup["pentad"] = step_lookup["date_min"].dt.dayofyear.sub(1).floordiv(5).add(1)
step_lookup = step_lookup[["year", "step8", "pentad"]].drop_duplicates()

era_p = era_p.merge(step_lookup, on=["year", "pentad"], how="inner")
merged = trc_p.merge(
    era_p.drop(columns=["pentad"]),
    on=["uc_id", "year", "step8"],
    how="left"
)

core = merged[merged["unit"] == "core"].copy()
core["rzsm_pct"] = core.groupby(["uc_id", "step8"])["rzsm"].transform(
    lambda x: x.rank(pct=True, method="average") * 100.0
)
core["vpd_pct"] = core.groupby(["uc_id", "step8"])["vpd_kpa"].transform(
    lambda x: x.rank(pct=True, method="average") * 100.0
)
core["t2m_pct"] = core.groupby(["uc_id", "step8"])["t2m_c"].transform(
    lambda x: x.rank(pct=True, method="average") * 100.0
)

merged = merged.merge(
    core[["uc_id", "year", "step8", "rzsm_pct", "vpd_pct", "t2m_pct"]].drop_duplicates(),
    on=["uc_id", "year", "step8"],
    how="left"
)
print("Merged panel shape:", merged.shape)


In [ ]:
city_unit_cov = (
    merged.groupby(["uc_id", "unit"], as_index=False)
          .agg(
              day_valid_frac=("dT_day", lambda x: x.notna().mean()),
              night_valid_frac=("dT_night", lambda x: x.notna().mean()),
          )
)

wide = city_unit_cov.pivot(index="uc_id", columns="unit", values=["day_valid_frac", "night_valid_frac"])
wide.columns = ["_".join(col) for col in wide.columns]
wide = wide.reset_index()

wide["retain_day_fullperiod"] = (
    (wide["day_valid_frac_core"] >= FULL_PERIOD_FRAC) &
    (wide["day_valid_frac_ring10_20"] >= FULL_PERIOD_FRAC)
)
wide["retain_night_fullperiod"] = (
    (wide["night_valid_frac_core"] >= FULL_PERIOD_FRAC) &
    (wide["night_valid_frac_ring10_20"] >= FULL_PERIOD_FRAC)
)
wide["retain_either_fullperiod"] = wide["retain_day_fullperiod"] | wide["retain_night_fullperiod"]

keep_ids = set(wide.loc[wide["retain_either_fullperiod"], "uc_id"].astype(str))
panel = merged[merged["uc_id"].isin(keep_ids)].copy()

ret_2025 = pd.read_csv(RETENTION_2025, dtype={"uc_id": str})
ret_2025["city_retain_either"] = ret_2025["city_retain_either"].astype(str).str.lower().isin(["true", "1", "yes"])

city_base = (
    panel[panel["unit"] == "ring10_20"]
    .groupby("uc_id", as_index=False)
    .agg(mean_ring_veg=("n_veg_day", "mean"))
)
veg_median = city_base["mean_ring_veg"].median()
city_base["veg_group"] = np.where(city_base["mean_ring_veg"] >= veg_median, "high_veg", "low_veg")

panel = panel.merge(city_base[["uc_id", "mean_ring_veg", "veg_group"]], on="uc_id", how="left")

flow_df = pd.DataFrame(
    [
        {"stage": "screened_2025_ids", "cities": int(ret_2025["city_retain_either"].sum())},
        {"stage": "fullperiod_retained", "cities": len(keep_ids)},
        {"stage": "high_veg_main_subgroup", "cities": int((city_base["veg_group"] == "high_veg").sum())},
    ]
)
flow_df


In [ ]:
clim = (
    panel.groupby(["uc_id", "unit", "step8"], as_index=False)
         .agg(
             dT_day_clim=("dT_day", "mean"),
             dT_night_clim=("dT_night", "mean"),
             tb_day_clim=("tb_day", "mean"),
             tb_night_clim=("tb_night", "mean"),
             tv_day_clim=("tv_day", "mean"),
             tv_night_clim=("tv_night", "mean"),
         )
)

panel = panel.merge(clim, on=["uc_id", "unit", "step8"], how="left")
panel["dT_day_anom"] = panel["dT_day"] - panel["dT_day_clim"]
panel["dT_night_anom"] = panel["dT_night"] - panel["dT_night_clim"]
panel["tb_day_anom"] = panel["tb_day"] - panel["tb_day_clim"]
panel["tb_night_anom"] = panel["tb_night"] - panel["tb_night_clim"]
panel["tv_day_anom"] = panel["tv_day"] - panel["tv_day_clim"]
panel["tv_night_anom"] = panel["tv_night"] - panel["tv_night_clim"]

panel_parquet = OUT / "TRC_threshold_analysis_panel_anom_corrected.parquet"
panel.to_parquet(panel_parquet, index=False)

if WRITE_PANEL_CSV_GZ:
    panel.to_csv(OUT / "TRC_threshold_analysis_panel_anom_corrected.csv.gz", index=False, compression="gzip")

wide.to_csv(OUT / "city_retention_fullperiod_corrected.csv", index=False)
flow_df.to_csv(OUT / "sample_flow_corrected.csv", index=False)
city_base.to_csv(OUT / "panel_city_base_corrected.csv", index=False)

print("Wrote:", panel_parquet)
print("Wrote:", OUT / "city_retention_fullperiod_corrected.csv")
print("Wrote:", OUT / "sample_flow_corrected.csv")
print(flow_df)
